# PyGeoFetch — 01: Getting Started

**PyGeoFetch** is a universal satellite data pipeline: 22+ providers, 17 spectral
indices, SAR processing, and chainable YAML pipelines — in one CLI and Python API.

### What you'll learn
- Install and verify PyGeoFetch
- Run your first search (no credentials needed)
- Download your first scene (RGB bands only)
- Explore results as a pandas DataFrame
- Use `client.status()` to see your setup at a glance

## 1. Installation

In [1]:
# Install PyGeoFetch
# Uncomment once and run — then restart the kernel

# !pip install PyGeoFetch           # core only (free providers work immediately)
# !pip install "PyGeoFetch[geo]"    # +rasterio, geopandas, shapely
# !pip install "PyGeoFetch[all]"    # everything including dev tools

# Verify
import pygeofetch
print(f"PyGeoFetch {pygeofetch.__version__} imported successfully")

PyGeoFetch 1.0.7 imported successfully


In [2]:
# Doctor — full installation diagnostic
# Checks Python version, packages, keyring, and live connectivity to free providers
!PyGeoFetch doctor


PyGeoFetch Doctor v1.0.0

  ✓ Python 3.12.3
  ✓ httpx
  ✓ pydantic
  ✓ click
  ✓ rich
  ✓ yaml
  ✓ cryptography
  ✓ keyring
  ✓ boto3 (optional: AWS S3 direct access)
  ⚠ rasterio not installed — raster post-processing unavailable
  ⚠ geopandas not installed — GeoParquet output unavailable
  ⚠ croniter not installed — cron scheduling unavailable
  ✓ Config directory: /home/mrtenkorang/.pygeofetch
  ✓ Keyring backend: Keyring

  Checking provider connectivity...
  ✓ AWS Earth Search: HTTP 200
  ✓ Planetary Computer: HTTP 200
  ✓ Element 84: HTTP 200

Doctor complete.


## 2. Free Providers — No Credentials Needed

In [4]:
from pygeofetch.providers import list_provider_info, get_free_providers
import pandas as pd

all_providers = list_provider_info()
free_ids      = set(get_free_providers())

rows = []
for p in all_providers:
    rows.append({
        "ID":          p["id"],
        "Free?":       "✓" if p["id"] in free_ids else "—",
        "SAR":         "✓" if p.get("sar") else "—",
        "STAC":        "✓" if p.get("stac") else "—",
        "VHR (<1m)":   "✓" if p.get("sub_meter") else "—",
        "Description": p.get("name", ""),
    })

df = pd.DataFrame(rows)
print(f"Total providers: {len(df)}  |  Free (no login): {df['Free?'].eq('✓').sum()}")
print()
print(df[df["Free?"] == "✓"].to_string(index=False))

Total providers: 23  |  Free (no login): 10

                ID Free? SAR STAC VHR (<1m)                  Description
         aws_earth     ✓   —    ✓         —             AWS Earth Search
      digitalglobe     ✓   —    —         —       DigitalGlobe Open Data
         element84     ✓   —    ✓         —      Element 84 Earth Search
        esa_scihub     ✓   —    —         —    ESA Copernicus Hub Mirror
 geoserver_generic     ✓   —    —         —            GeoServer Generic
        inpe_cbers     ✓   —    —         —                   INPE CBERS
       isro_bhuvan     ✓   —    —         —                  ISRO Bhuvan
        jaxa_earth     ✓   —    —         —              JAXA ALOS World
     noaa_big_data     ✓   —    —         —                NOAA Big Data
planetary_computer     ✓   —    ✓         — Microsoft Planetary Computer


## 3. Your First Search — Python API

In [5]:
from pygeofetch import PyGeoFetch
from pygeofetch.models.search_query import SearchQuery, BoundingBox
from pathlib import Path

Path("output").mkdir(exist_ok=True)

# Initialise client (WARNING level keeps output clean in notebooks)
client = PyGeoFetch(log_level="INFO")

# NYC bounding box — longitude first
nyc_bbox = BoundingBox.from_string("-74.1,40.6,-73.7,40.9")

query = SearchQuery(
    bbox=nyc_bbox,
    start_date="2024-01-01",
    end_date="2024-06-01",
    cloud_cover_max=15,
    max_results=50,
    sort_by="cloud_cover",
    sort_ascending=True,
)

results = client.search(query, providers=["aws_earth", "planetary_computer"])
print(f"\nFound {len(results)} scenes")

19:54:53 INFO [    engine] PyGeoFetch ready
19:54:53 INFO [  searcher] Searching 2 provider(s): aws_earth, planetary_computer
19:54:55 INFO [planetary_computer] Planetary Computer: 50 results
19:54:55 INFO [  searcher]   planetary_computer: 50 results
19:54:55 INFO [  searcher]   aws_earth: 50 results
19:54:55 INFO [  searcher] Search complete: 50 unique results from 2 providers

Found 50 scenes


## 4. Explore Results

In [6]:
import pandas as pd

records = []
for r in results:
    records.append({
        "id":         r.id[:40],
        "provider":   r.provider,
        "satellite":  r.satellite or "—",
        "date":       str(r.datetime)[:10] if r.datetime else "—",
        "cloud_pct":  f"{r.cloud_cover:.1f}" if r.cloud_cover is not None else "—",
        "score":      f"{r.score:.2f}",
        # New SAR/product fields (populated for SAR products)
        "product_type":  r.product_type or "—",
        "polarisation":  r.polarisation or "—",
    })

df = pd.DataFrame(records)
print("Top 10 clearest scenes:")
print(df.head(10).to_string(index=False))

Top 10 clearest scenes:
                                      id           provider   satellite       date cloud_pct score product_type polarisation
S2A_MSIL2A_20240416T153941_R011_T18TXK_2 planetary_computer Sentinel-2A 2024-04-16       0.0  0.70            —            —
S2A_MSIL2A_20240426T153941_R011_T18TXK_2 planetary_computer Sentinel-2A 2024-04-26       0.0  0.70            —            —
S2A_MSIL2A_20240416T153941_R011_T18TWL_2 planetary_computer Sentinel-2A 2024-04-16       0.1  0.70            —            —
S2B_MSIL2A_20240325T154809_R054_T18TWK_2 planetary_computer Sentinel-2B 2024-03-25       0.1  0.70            —            —
                S2B_18TXL_20240524_0_L2A          aws_earth sentinel-2b 2024-05-24       0.1  0.70            —            —
S2B_MSIL2A_20240524T154809_R054_T18TXL_2 planetary_computer Sentinel-2B 2024-05-24       0.1  0.70            —            —
         LC09_L2SP_013032_20240312_02_T1 planetary_computer   landsat-9 2024-03-12       0.1  0.70   

In [7]:
# Inspect one scene in detail
scene = results[0]
print(f"Scene ID:      {scene.id}")
print(f"Provider:      {scene.provider}")
print(f"Satellite:     {scene.satellite}")
print(f"Date/Time:     {scene.datetime}")
print(f"Cloud cover:   {scene.cloud_cover}%")
print(f"Product type:  {scene.product_type or 'N/A'}")
print(f"Polarisation:  {scene.polarisation or 'N/A'}")
print(f"Pass dir:      {scene.pass_direction or 'N/A'}")
print(f"GSD (m):       {scene.gsd_m or 'N/A'}")
print(f"Score:         {scene.score:.2f}")
print(f"Assets:        {list(scene.assets.keys())[:8]}")

Scene ID:      S2A_MSIL2A_20240416T153941_R011_T18TXK_20240417T002018
Provider:      planetary_computer
Satellite:     Sentinel-2A
Date/Time:     2024-04-16 15:39:41.024000+00:00
Cloud cover:   0.014937%
Product type:  N/A
Polarisation:  N/A
Pass dir:      descending
GSD (m):       N/A
Score:         0.70
Assets:        ['AOT', 'B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07']


In [8]:
# Save results to GeoJSON — used in notebook 04 for downloads
client.searcher.save_results(results, Path("output/nyc_results.geojson"))
print(f"✓ Saved {len(results)} results → output/nyc_results.geojson")

19:55:53 INFO [  searcher] Saved 50 results to output/nyc_results.geojson
✓ Saved 50 results → output/nyc_results.geojson


## 5. Your First Download

In [9]:
from pygeofetch.models.download_task import DownloadOptions

best = results[0]
print(f"Downloading: {best.id}")
print(f"Cloud cover: {best.cloud_cover:.1f}%")

options = DownloadOptions(
    parallel=1,
    retry_attempts=3,
    resume=True,
    verify_checksum=False,
    on_failure="skip",
    bands=["B02", "B03", "B04"],   # RGB only — ~150 MB vs ~600 MB full scene
)

dl_results = client.download(
    [best],
    destination=Path("output/downloads/"),
    options=options,
)

for dr in dl_results:
    if dr.success:
        size_mb = dr.bytes_downloaded / 1024 / 1024
        print(f"\n✓ Downloaded {dr.data_id} ({size_mb:.1f} MB)")
    else:
        print(f"\n✗ Failed: {dr.error}")

Downloading: S2A_MSIL2A_20240416T153941_R011_T18TXK_20240417T002018
Cloud cover: 0.0%
19:56:06 INFO [downloader] Downloading 1 scene(s) to: output/downloads
19:56:06 INFO [downloader] Downloading 'S2A_MSIL2A_20240416T153941_R011_T18TXK_20240417T002018' from 'planetary_computer' → output/downloads/planetary_computer
19:56:07 INFO [planetary_computer] Downloading 3 asset(s) for S2A_MSIL2A_20240416T153941_R011_T18TXK_20240417T002018: ['B02', 'B03', 'B04']
19:56:07 INFO [planetary_computer]   Fetching asset 'B02' → T18TXK_20240416T153941_B02_10m.tif
19:58:51 INFO [planetary_computer]   ✓ T18TXK_20240416T153941_B02_10m.tif (164.3 MB)
19:58:51 INFO [planetary_computer]   Fetching asset 'B03' → T18TXK_20240416T153941_B03_10m.tif
20:00:40 INFO [planetary_computer]   ✓ T18TXK_20240416T153941_B03_10m.tif (162.0 MB)
20:00:40 INFO [planetary_computer]   Fetching asset 'B04' → T18TXK_20240416T153941_B04_10m.tif
20:19:45 INFO [planetary_computer]   ✓ T18TXK_20240416T153941_B04_10m.tif (158.4 MB)
20:

## 6. System Status

In [ ]:
status = client.status()
print(f"PyGeoFetch v{status['version']}")
print(f"Authenticated providers : {status.get('providers_authenticated') or 'none'}")
print(f"Free providers ready    : {len(status.get('providers_free', []))}")
print(f"Cache entries           : {status.get('cache_entries', 0)}")
print(f"Cache hit rate          : {status.get('cache_hit_rate', 0):.0%}")

---
## Summary
- ✅ Installed and verified PyGeoFetch  
- ✅ Listed 22 providers — 10 need no login  
- ✅ Searched AWS Earth + Planetary Computer (free) — federated, deduplicated  
- ✅ Inspected all fields including new SAR fields (product_type, polarisation, pass_direction)  
- ✅ Downloaded RGB bands only (75% smaller than full scene)  

**Next:** Notebook 02 — Authentication & all 22 providers